# 슈퍼SOL 고객 반응 분석 노트북

수집 → 전처리 → 감성 → 이슈/토픽 → 경쟁 벤치마킹 흐름과 판단 근거를 정리한다.

- 실행 전: `pip install -r ../requirements.txt`
- 데이터가 없으면: 루트에서 `python -m src.pipeline` 1회 실행(또는 `sample_data/sample_reviews.csv` 사용)
- ⚠️ 감성 라벨은 **별점 기반 약지도**(실제 감정 정답 아님). 답글은 **검수용 초안**.

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))  # 루트 모듈 import
import pandas as pd
from src.config import PATHS
pd.set_option('display.max_colwidth', 60)

## 1. 데이터 로드 & 기초 EDA

In [ ]:
df = pd.read_csv(PATHS.processed_dir / 'review_clean.csv')
df['date'] = pd.to_datetime(df['date'], errors='coerce')
print('총 리뷰:', len(df), '| 앱:', df['app_name'].nunique())
df[['app_name','store','rating','date','clean_text']].head()

In [ ]:
# 앱별 리뷰 수 / 평균 평점 / 부정률
g = df.groupby('app_name').agg(리뷰수=('rating','size'), 평균평점=('rating','mean'))
g['부정률'] = df.groupby('app_name')['rating'].apply(lambda s: (s<=2).mean())
g.round(3).sort_values('평균평점', ascending=False)

## 2. 전처리 판단 근거
- 결측·중복·초단문 제거, 개인정보 [MASKED], 날짜 표준화, 길이 IQR 이상치 표시
- 상세 지표는 `outputs/data_quality_summary.json`

In [ ]:
import json
print(json.dumps(json.load(open(PATHS.outputs_dir / 'data_quality_summary.json', encoding='utf-8')), ensure_ascii=False, indent=2))

## 3. 감성 분석 (ML, 별점 약지도 라벨)
- 라벨: 1~2=neg, 3=neu, 4~5=pos (실제 감정 정답 아님)
- baseline(Dummy / TF-IDF+LogReg) → 개선(ngram·class_weight)

In [ ]:
from src.train_sentiment import make_weak_label
df['label'] = pd.to_numeric(df['rating'], errors='coerce').dropna().astype(int).map(make_weak_label)
print('약지도 라벨 분포:', df['label'].value_counts().to_dict())
m = json.load(open(PATHS.outputs_dir / 'metrics.json', encoding='utf-8'))
print('baseline:', {k: v['f1_macro'] for k,v in m['sentiment_baseline']['models'].items()})
print('final f1(별점기준):', m['final_model']['f1_macro_star'])

## 4. 이슈 유형(룰) & 토픽(비지도)

In [ ]:
from src.issue_classifier import classify_issue_type, classify_severity
ex = df[df['rating']<=2]['clean_text'].head(5)
for t in ex:
    it = classify_issue_type(t)
    print(classify_severity(t, 1, it), it, '|', str(t)[:40])

In [ ]:
pd.read_csv(PATHS.outputs_dir / 'topic_summary.csv')[['topic_id','size','auto_label','top_keywords']]

## 5. 경쟁 앱 벤치마킹

In [ ]:
pd.read_csv(PATHS.outputs_dir / 'benchmark_summary.csv')[['app_name','review_count','avg_rating','negative_ratio','top_complaint_types']]

## 6. 결론 & 한계
- 슈퍼SOL은 경쟁 대비 중하위 평점, 공통 최다 불만 = 오류 → 안정성 개선이 핵심.
- 한계: 별점 약지도 라벨, App Store RSS 수집량 상한/시점 편중.
- 고도화: 수동 검수 라벨, 벡터 RAG, 임베딩 감성 모델, 정기 자동 수집.
- 답글 초안은 **검수용**이며 자동 게시하지 않음.